# 03 · Public help-center scraping

**Portfolio adaptation of the original 2023 `TA_Help_File_Scrape.ipynb`.** Demonstrates public help-center discovery, collection/article URL extraction, H2/H3 content extraction, and CSV export.

This revision separates scraping functions from file I/O and corrects URL joining and missing-article handling. The source notebook is under `notebooks/sanitized_reference/`. It defaults to **synthetic offline HTML fixtures**; optional live access is strictly opt-in and should comply with website permissions, rate limits, and terms.

In [ ]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
import csv
import os
import time

from bs4 import BeautifulSoup

PROJECT_DIR = Path(os.environ.get("SJK_PORTFOLIO_DIR", "/content/sjk_portfolio"))
OUTPUT_DIR = PROJECT_DIR / "help_center"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEMO_MODE = os.environ.get("SJK_DEMO_MODE", "1") == "1"
LIVE_BASE_URL = "https://help.typingagent.com/en/"
print("Output directory:", OUTPUT_DIR, "Demo mode:", DEMO_MODE)


## Public help-page discovery and article extraction
Maintains the three original steps: collections → articles → rows containing article headings and content. Tests replace the HTTP client; they do not access the web.


In [ ]:
def matching_links(html, page_url, fragment, allowed_host):
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    for item in soup.select("a[href]"):
        resolved = urljoin(page_url, item["href"])
        parsed = urlparse(resolved)
        if parsed.scheme in ("http", "https") and parsed.hostname == allowed_host and fragment in parsed.path:
            if resolved not in urls:
                urls.append(resolved)
    return urls


def extract_article_rows(html, article_url):
    soup = BeautifulSoup(html, "html.parser")
    article = soup.find("article")
    if article is None:
        return []
    title_meta = soup.find("meta", attrs={"property": "og:title"})
    description_meta = soup.find("meta", attrs={"property": "og:description"})
    title = title_meta.get("content", "") if title_meta else ""
    description = description_meta.get("content", "") if description_meta else ""
    heading = soup.find("h1", class_="page-title")
    subject = heading.get_text(" ", strip=True) if heading else title
    rows = []
    for section in article.find_all(["h2", "h3"]):
        pieces = []
        for sibling in section.next_siblings:
            if getattr(sibling, "name", "") in ("h2", "h3"):
                break
            if hasattr(sibling, "get_text"):
                pieces.append(sibling.get_text(" ", strip=True))
            elif isinstance(sibling, str):
                pieces.append(sibling.strip())
        rows.append({"title": title, "href": article_url,
                     "description": description, "subject": subject,
                     "h2": section.get_text(" ", strip=True),
                     "article": " ".join(part for part in pieces if part)})
    return rows


def run_help_scrape(fetch_text, base_url, output_dir=OUTPUT_DIR, max_collections=3, max_articles=10):
    """Use an injected HTML fetcher; callers control any network access."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    host = urlparse(base_url).hostname
    collections = matching_links(fetch_text(base_url), base_url, "/collections/", host)
    articles = []
    for collection in collections[:max_collections]:
        for url in matching_links(fetch_text(collection), collection, "/articles/", host):
            if url not in articles:
                articles.append(url)
    rows = []
    for url in articles[:max_articles]:
        rows.extend(extract_article_rows(fetch_text(url), url))
    files = [("collections.csv", [[u] for u in collections]),
             ("articles.csv", [[u] for u in articles])]
    for filename, values in files:
        with (output_dir / filename).open("w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerows(values)
    result = output_dir / "help_articles.csv"
    with result.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["title", "href", "description", "subject", "h2", "article"])
        writer.writeheader()
        writer.writerows(rows)
    return {"collections": len(collections), "articles": len(articles), "sections": len(rows), "output": str(result)}


## Demo: synthetic pages only
Uses a made-up domain and small page fixtures. When running live, the fetcher has a timeout, raises on HTTP errors, and pauses between requests; you must independently verify scraping permission and applicable terms.


In [ ]:
if DEMO_MODE:
    demo_base = "https://help.example.org/en/"
    pages = {
      demo_base: '<a href="/en/collections/demo-guides">Guides</a>',
      "https://help.example.org/en/collections/demo-guides": '<a href="/en/articles/demo-setup">Setup article</a>',
      "https://help.example.org/en/articles/demo-setup": """<html><head><meta property="og:title" content="Example setup" />
      <meta property="og:description" content="Synthetic article example" /></head>
      <body><h1 class="page-title">Example setup</h1><article><h2>Getting started</h2>
      <p>Open the synthetic dashboard.</p><h3>Next steps</h3><p>Continue with a test account.</p>
      </article></body></html>"""
    }
    result = run_help_scrape(lambda url: pages[url], demo_base)
else:
    import requests
    session = requests.Session()
    def fetch_live(url):
        response = session.get(url, timeout=15, headers={"User-Agent": "PortfolioEducationalExample/1.0"})
        response.raise_for_status()
        time.sleep(1)
        return response.text
    result = run_help_scrape(fetch_live, LIVE_BASE_URL)
print(result)


## Limits
This verifies HTML parsing, public-URL normalization, de-duplication, and CSV export against synthetic pages. It **does not confirm the 2026 structure or scraping permissions of the real help center.** Do not commit scraped pages without checking reuse rights and personal data.
